# Pareja C — Cierre: del registro al modelo final (versión Kaggle)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/11909/Poster2Movie-Clasificador/blob/main/notebooks/C_cierre.ipynb)

Lo mismo que `C_cierre.ipynb` (Colab), pero en Kaggle: sirve para usar la GPU de Kaggle (30 h por
semana) en los pasos 1 y 7 cuando se acaba la de Colab.

Junta lo que funcionó de todas las ideas, arma el modelo final y lo deja listo para el release. **Solo
validación**: test se mide una sola vez, en `C_release`.

Estado al 27 nov (validación, base: F1 macro 0.544, promedio de los 4 débiles 0.384):

| Experimento | F1 macro | Débiles | |
| --- | --- | --- | --- |
| `A7_l14` (CLIP L/14) | 0.605 (+0.061) | 0.429 (+0.045) | el salto grande |
| `A1_3recortes` (B/32) | 0.563 (+0.019) | 0.391 (+0.008) | ayuda: probarlo en L/14 |
| `A7_siglip` | 0.560 (+0.016) | 0.379 (−0.005) | diversidad para el ensamble |
| `B_combinado_sinpw` (B/32, 4 semillas) | 0.544 a 0.555 | 0.374 a 0.399 | mejora dentro del ruido: probarla en L/14 |
| `A2_ocr` | — | — | solo 195 películas de val (muestra): no comparable, no entra al ensamble |

Orden: (1) vectores L/14 con 3 recortes (GPU) → (2) experimentos nuevos sobre L/14 → (3) miembros →
(4) ensambles → (5) apilamiento → (6) tabla final → (7) memoria en T4 → (8) release con `C_release`.
Sin GPU se saltan solos los pasos 1 y 7; el resto corre en CPU.

**Diferencia con Colab:** Kaggle no puede montar Google Drive. Este notebook usa
[rclone](https://rclone.org/drive/) para:

1. **Bajar** del Drive lo necesario: la metadata del dataset, los vectores, modelos y predicciones de
   la carpeta del equipo y, solo si hay que calcular el paso 1, los carteles (~3.3 GB).
2. **Subir solo** al Drive del equipo (`poster2movie_clasificador/`) los vectores, modelos y
   predicciones nuevos: cada 10 minutos y al terminar cada paso. **Nunca borra** nada del Drive.

Las métricas (`resultados/`) se suben a la rama `C-cierre` con la última celda, como en Colab (y
también se copian al Drive como respaldo).

## Antes de la primera vez

**En Kaggle** (panel derecho, *Settings*): *Accelerator* → **GPU T4 x2** (solo para los pasos 1 y 7; sin GPU el resto corre igual) e *Internet* →
**On** (pide verificar la cuenta con un teléfono).

**Llave del Drive (una vez por persona):** en tu computadora instala rclone
(<https://rclone.org/downloads/>) y corre:

```bash
rclone authorize "drive"
```

Se abre el navegador: entra con la cuenta de Google que tiene `poster2movie_datos` y
`poster2movie_clasificador` como acceso directo en *Mi unidad*. Al final la terminal imprime un texto
`{"access_token": ..., "refresh_token": ..., ...}`: cópialo completo (una línea, entre las llaves).

**En Kaggle**, *Add-ons → Secrets*:

| Secreto | Valor | ¿Obligatorio? |
| --- | --- | --- |
| `RCLONE_DRIVE_TOKEN` | el texto `{...}` de `rclone authorize` | sí |
| `GITHUB_TOKEN` | token *classic* con permiso `repo` (el mismo de Colab) | para subir `resultados/` sin que lo pida |
| `GIT_NOMBRE`, `GIT_CORREO` | tu nombre y correo de GitHub | igual |

Activa los secretos en **este** notebook (casilla junto a cada uno).

⚠️ El `RCLONE_DRIVE_TOKEN` da acceso a **todo tu Drive**: guárdalo solo en *Secrets* de Kaggle, nunca
en una celda, en el repo ni en el chat. Si se filtra, quítale el acceso a rclone en
<https://myaccount.google.com/permissions>.

**Correr sin tener la pestaña abierta:** *Save Version → Save & Run All (Commit)*. Corre en segundo
plano hasta 12 h; con los secretos `GITHUB_TOKEN`, `GIT_NOMBRE` y `GIT_CORREO` sube los resultados
solo. Si se corta, vuelve a correrlo: continúa con los pedazos que ya están en el Drive.

## Preparar el entorno

Cambia las carpetas si en tu *Mi unidad* los accesos directos tienen otro nombre.

In [ ]:
import json, os, subprocess, threading, time
from pathlib import Path

# --- Drive del equipo (rutas dentro de *Mi unidad*) ---
REMOTO = "gdrive"                          # nombre interno de rclone, no hay que cambiarlo
DRIVE_DATOS = "poster2movie_datos"          # dataset compartido (solo se LEE)
DRIVE_SALIDAS = "poster2movie_clasificador"  # carpeta del equipo (aquí se SUBEN vectores, modelos, predicciones)

# --- Repo ---
REPO = "11909/Poster2Movie-Clasificador"
RAMA = "C-cierre"
DESTINO = "/kaggle/working/clasificador"   # en /kaggle/working: queda en la salida del notebook

# --- Carpetas locales de Kaggle ---
os.environ["P2M_DATOS"] = "/tmp/p2m_datos"                  # copia local del dataset (no va a la salida)
os.environ["P2M_SALIDAS"] = "/kaggle/working/p2m_salidas"   # se sincroniza con DRIVE_SALIDAS
for carpeta in (os.environ["P2M_DATOS"], os.environ["P2M_SALIDAS"]):
    Path(carpeta).mkdir(parents=True, exist_ok=True)
DATOS, SALIDAS = os.environ["P2M_DATOS"], os.environ["P2M_SALIDAS"]
DRIVE = f"{REMOTO}:{DRIVE_SALIDAS}"
EN_SEGUNDO_PLANO = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") == "Batch"   # "Save & Run All"


def secreto(nombre):
    # Un secreto de Kaggle (Add-ons -> Secrets), o None si no existe o no está activado.
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(nombre)
    except Exception:
        return None


# --- rclone: instalar y configurar con la llave del Drive ---
if not Path("/usr/local/bin/rclone").exists():
    subprocess.run("curl -sSL -o /tmp/rclone.zip https://downloads.rclone.org/rclone-current-linux-amd64.zip"
                   " && unzip -q -o /tmp/rclone.zip -d /tmp/rclone"
                   " && cp /tmp/rclone/rclone-*-linux-amd64/rclone /usr/local/bin/", shell=True, check=True)

token = secreto("RCLONE_DRIVE_TOKEN")
if not token:
    raise RuntimeError("Falta el secreto RCLONE_DRIVE_TOKEN (ver 'Antes de la primera vez'): "
                       "Add-ons -> Secrets, y activarlo para este notebook.")
json.loads(token)  # si falla aquí, el secreto no es el texto {...} completo de `rclone authorize "drive"`
conf = Path.home() / ".config/rclone/rclone.conf"
conf.parent.mkdir(parents=True, exist_ok=True)
conf.write_text(f"[{REMOTO}]\ntype = drive\nscope = drive\ntoken = {token.strip()}\n")
conf.chmod(0o600)
del token


def rclone(*args, check=True):
    return subprocess.run(["rclone", *args], check=check, capture_output=True, text=True)


for carpeta in (DRIVE_DATOS, DRIVE_SALIDAS):
    r = rclone("lsf", f"{REMOTO}:{carpeta}", "--max-depth", "1", check=False)
    if r.returncode != 0:
        raise RuntimeError(f"No encuentro '{carpeta}' en tu Mi unidad. ¿Agregaste el acceso directo? "
                           f"Detalle de rclone:\n{r.stderr}")
print("Drive conectado:", DRIVE_DATOS, "y", DRIVE_SALIDAS)

def bajar_vectores(nombres):
    # Trae del Drive los .npz (y su .json) de cada vector; los que ya estén aquí no se vuelven a bajar.
    # Orden de búsqueda igual que `rutas.ruta_vectores`: carpeta del equipo, dataset compartido, alias de la base.
    for nombre in nombres:
        destino = Path(SALIDAS) / "vectores"
        if (destino / f"{nombre}.npz").exists():
            continue
        if rclone("lsf", f"{DRIVE}/vectores/{nombre}.npz", check=False).stdout.strip():
            rclone("copy", f"{DRIVE}/vectores", str(destino), "--include", f"/{nombre}.*")
        elif rclone("lsf", f"{REMOTO}:{DRIVE_DATOS}/vectores/{nombre}.npz", check=False).stdout.strip():
            rclone("copy", f"{REMOTO}:{DRIVE_DATOS}/vectores", f"{DATOS}/vectores", "--include", f"/{nombre}.*")
        elif nombre == "clip_b32__centro":   # los vectores de la base viven en el dataset compartido
            rclone("copy", f"{REMOTO}:{DRIVE_DATOS}", DATOS, "--include", "/embeddings_clip_b32.npz")
        else:
            print(f"Aviso: no encuentro los vectores '{nombre}' en el Drive.")
            continue
        print("Bajado:", nombre)


In [ ]:
# --- Repo y dependencias ---
if not os.path.isdir(DESTINO):
    !git clone -q --recurse-submodules https://github.com/{REPO}.git {DESTINO}
%cd {DESTINO}
!git fetch -q origin
!git checkout -q {RAMA} 2>/dev/null || git checkout -q -b {RAMA} origin/main
!git pull -q origin {RAMA} 2>/dev/null; git submodule update -q --init
!pip install -q -r requirements.txt
!nvidia-smi -L 2>/dev/null || echo "Sin GPU (basta para todo salvo los pasos que la piden)"

## Bajar del Drive y sincronizar

La metadata del dataset (los carteles solo si hace falta el paso 1), y de la carpeta del equipo los
vectores, modelos y predicciones (unos GB: L/14 y los 3 recortes son lo más pesado). Después, un hilo
sube lo nuevo al Drive cada 10 minutos.

In [ ]:
print("Bajando la metadata del dataset...")
rclone("copy", f"{REMOTO}:{DRIVE_DATOS}", DATOS, "--include", "/metadata.parquet")
bajar_vectores(["clip_b32__centro"])

print("Bajando vectores, modelos y predicciones del equipo...")
rclone("copy", DRIVE, SALIDAS, "--include", "/vectores/*", "--include", "/modelos/**",
       "--include", "/predicciones/**", "--transfers", "8")
for carpeta in ("vectores", "modelos", "predicciones"):
    print(f"{carpeta}: {sorted(p.name for p in Path(SALIDAS, carpeta).glob('*'))}")

_candado = threading.Lock()
_detener = threading.Event()


def subir_al_drive():
    # vectores/, modelos/ y predicciones/ locales -> carpeta del equipo en el Drive (nunca borra nada).
    with _candado:
        r = rclone("copy", SALIDAS, DRIVE, "--transfers", "4", check=False)
    if r.returncode != 0:
        print("Aviso: falló la subida al Drive (se reintenta en la siguiente):", r.stderr[-500:])


def subir_resultados(nombres):
    # resultados/<nombre> del repo -> Drive, como respaldo (en GitHub van con la última celda).
    for nombre in nombres:
        if Path(f"resultados/{nombre}").exists():
            rclone("copy", f"resultados/{nombre}", f"{DRIVE}/resultados/{nombre}", check=False)


def _sincronizar(cada=600):
    while not _detener.wait(cada):
        subir_al_drive()


threading.Thread(target=_sincronizar, daemon=True).start()
print("Sincronización con el Drive cada 10 minutos: activa.")

## 1. Vectores de CLIP L/14 con 3 recortes (GPU, opcional)

Tres pasadas de L/14 por cartel: lo más caro del cierre. Si ya están completos en el Drive no se
recalculan; si alguien empezó, continúa con los pedazos que ya están. Sin GPU se salta.

In [ ]:
import torch
NOMBRE_3R = "clip_l14__3recortes"
if Path(SALIDAS, "vectores", f"{NOMBRE_3R}.npz").exists():
    print(f"{NOMBRE_3R}: ya está completo en el Drive, no se recalcula.")
elif not torch.cuda.is_available():
    print(f"Sin GPU: salto {NOMBRE_3R} (A1_l14_3recortes se saltará en el paso 2).")
else:
    print("Bajando los carteles (~3.3 GB)...")
    rclone("copy", f"{REMOTO}:{DRIVE_DATOS}", DATOS, "--include", "/*.parquet", "--transfers", "8")
    rclone("copy", f"{DRIVE}/vectores/partes/{NOMBRE_3R}", f"{SALIDAS}/vectores/partes/{NOMBRE_3R}", check=False)
    hechos = len(list(Path(f"{SALIDAS}/vectores/partes/{NOMBRE_3R}").glob("*.npz")))
    print(f"{NOMBRE_3R}: {hechos} pedazos ya calculados en el Drive.")
    !python -m clasificador.codificadores clip_l14 --variante 3recortes
    subir_al_drive()

## 2. Experimentos nuevos sobre L/14 (CPU)

- `A1_l14_3recortes`: idea 1 sobre el mejor codificador.
- `B_l14_combinado_sinpw` (+ `_s1`, `_s2`): la mejor receta de B (MLP + focal) sobre L/14, tres semillas.

El que no tenga sus vectores se salta (p. ej. si no corrió el paso 1).

In [ ]:
from clasificador import rutas
from clasificador.entrenar import cargar_experimento

NUEVOS = ["A1_l14_3recortes", "B_l14_combinado_sinpw", "B_l14_combinado_sinpw_s1", "B_l14_combinado_sinpw_s2"]
for nombre in NUEVOS:
    try:
        for v in cargar_experimento(f"experimentos/{nombre}.yaml")["vectores"]:
            rutas.ruta_vectores(v)
    except FileNotFoundError as e:
        print(f"Salto {nombre}: {e}")
        continue
    !python -m clasificador.entrenar experimentos/{nombre}.yaml
    !python -m clasificador.evaluar {nombre}
!python -m clasificador.ranking A7_
!python -m clasificador.ranking A1_
!python -m clasificador.ranking B_l14

subir_al_drive()
subir_resultados(NUEVOS)

## 3. Miembros: el mejor experimento de cada codificador

`clasificador/cierre.py`: por codificador, el primero según `ranking` (cumple la regla > débiles > F1
macro). Deja fuera los combinados y los evaluados con menos películas que la base (OCR).

In [ ]:
from clasificador.cierre import mejores_por_codificador, COLUMNAS
mejores = mejores_por_codificador()
MIEMBROS = list(mejores.index)
print("MIEMBROS =", MIEMBROS)
mejores[COLUMNAS]

## 4. Ensambles (idea 7b)

- `C7_top2`: los dos mejores (por F1 macro).
- `C7_todos`: uno por codificador.
- `C7_todos_3recortes`: además `A1_3recortes` (B/32 viendo el cartel completo), si no está ya.

In [ ]:
ENSAMBLES = {"C7_top2": MIEMBROS[:2], "C7_todos": MIEMBROS}
if "A1_3recortes" not in MIEMBROS:
    ENSAMBLES["C7_todos_3recortes"] = MIEMBROS + ["A1_3recortes"]
for nombre, miembros in ENSAMBLES.items():
    if len(miembros) < 2:
        continue
    !python -m clasificador.ensamble {nombre} {" ".join(miembros)}
    !python -m clasificador.evaluar {nombre}

subir_al_drive()
subir_resultados(ENSAMBLES)

## 5. Apilamiento (idea 6)

Etapa 2 lineal residual con predicciones fuera de pliegue (5 pliegues por franquicia). Reentrena cada
miembro 5 veces: con L/14 y cabezas MLP tarda unos minutos por miembro (CPU basta). Cada uno deja
`resultados/<nombre>/pesos_etapa2.csv` y `mapa_calor.png` para el reporte.

In [ ]:
APILADOS = {"C6_mejor": MIEMBROS[:1], "C6_todos": MIEMBROS}
for nombre, miembros in APILADOS.items():
    !python -m clasificador.apilamiento {nombre} {" ".join(miembros)}
    !python -m clasificador.evaluar {nombre}

subir_al_drive()
subir_resultados(APILADOS)

## 6. Tabla final (validación)

Los miembros y todos los combinados contra la base. El candidato es el primero: cumple la regla
(macro no baja y los débiles suben) y tiene el mejor promedio de los débiles. Si dos están a menos de
~0.005 (el ruido entre semillas), elijan el más ligero (menos codificadores en la demo).

In [ ]:
from clasificador.cierre import tabla_completa
tabla = tabla_completa()
final = tabla[tabla.index.isin(MIEMBROS) | tabla.index.str.startswith("C")]
final = final.sort_values(["cumple", "debiles_prom", "F1_macro"], ascending=False)
final[COLUMNAS]

In [ ]:
from IPython.display import Image, display
CANDIDATO = final.index[0]
print("Candidato:", CANDIDATO)
!python -m clasificador.comparar base {CANDIDATO}
!python -m clasificador.comparar {MIEMBROS[0]} {CANDIDATO}
for nombre in APILADOS:
    if os.path.exists(f"resultados/{nombre}/mapa_calor.png"):
        display(Image(f"resultados/{nombre}/mapa_calor.png", width=500))

## 7. ¿Cabe en la demo? (GPU)

La demo corre en una T4 junto con la reseña (Qwen3-VL-2B) y el cartel (SD-Turbo). Si el candidato
no cabe, elijan el mejor de la tabla con menos codificadores. Sin GPU se salta.

In [ ]:
import torch
if not torch.cuda.is_available():
    print("Sin GPU: este paso se salta (la memoria se mide en una T4).")
else:
    from PIL import Image as PILImage
    from clasificador.predictor import Clasificador
    for nombre in final.index[:3]:
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
        clasificador = Clasificador(rutas.ruta_modelo(nombre))
        clasificador.predecir(PILImage.new("RGB", (384, 576)))
        print(f"{nombre:<24} pico de memoria GPU: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
        del clasificador

## 8. Release

Suban esta rama (celda de abajo) para que el registro y los mapas de calor queden en el repo. Luego
abran **`C_release`** (o `C_release_kaggle`) con `TAG = "v0.2.0"` y `EXPERIMENTO = "<candidato>"`:
mide val **y test** (la única vez que se mira test) y arma `releases/v0.2.0/`.

In [ ]:
subir_al_drive()
_detener.set()
print("Todo subido a", DRIVE_SALIDAS)

## Subir a la rama `C-cierre`

Se sube a **`C-cierre`**, nunca a `main`; luego abran el Pull Request (lo revisa alguien de
**otra** pareja). Con los secretos `GITHUB_TOKEN`, `GIT_NOMBRE` y `GIT_CORREO` no pregunta nada (sirve
en *Save & Run All*); sin ellos los pide aquí. Si corrió en segundo plano sin secretos, los resultados
quedan en la salida del notebook (`clasificador/`) y en el Drive (`resultados/`).

In [ ]:
from getpass import getpass

token = secreto("GITHUB_TOKEN")
nombre_git, correo_git = secreto("GIT_NOMBRE"), secreto("GIT_CORREO")
if EN_SEGUNDO_PLANO and not (token and nombre_git and correo_git):
    print("Sin GITHUB_TOKEN / GIT_NOMBRE / GIT_CORREO en segundo plano: no se sube a GitHub. "
          "Los resultados están en el Drive (resultados/) y en la salida del notebook.")
else:
    nombre_git = nombre_git or input("Tu nombre: ")
    correo_git = correo_git or input("Tu correo de GitHub: ")
    mensaje = f"Cierre: experimentos L/14, ensambles y apilamiento (desde Kaggle)" if EN_SEGUNDO_PLANO \
        else input("Qué hiciste (mensaje del commit): ")
    token = token or getpass("Token de GitHub: ")
    !git config user.name "{nombre_git}"
    !git config user.email "{correo_git}"
    !git add experimentos resultados
    !git commit -q -m "{RAMA}: {mensaje}"
    !git push -q https://{token}@github.com/{REPO}.git HEAD:{RAMA}
    print(f"Abre el Pull Request: https://github.com/{REPO}/compare/main...{RAMA}")
del token